# Weeks 2 & 3 Starter: Reward Engineering + Hyperparameter Ablation

**Starter Notebook — Weeks 2 & 3**

This notebook covers both topics in one workflow:

- **Part 1 (Reward Engineering):** Train Walker2D with four different reward functions and compare results.
- **Part 2 (Hyperparameter Ablation):** Fix the best reward function, vary one hyperparameter at a time, and measure the effect.
- **Part 3 (Custom Reward):** Design and test your own reward function.

**Deliverables:**
1. Four reward function training runs (300k steps each) with a comparison plot.
2. At least 5 ablation runs (300k steps each) with a comparison table.
3. One custom reward function with justification.
4. Written answers to all reflection questions.

> Read `week2_3_guide.pdf` before starting this notebook.  
> Run the solved examples notebook (`01_examples_reward_and_hyperparams.ipynb`) first to build intuition.

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import matplotlib.pyplot as plt

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecNormalize
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback
from env import Walker2DEnv, get_reward_fn, REWARD_REGISTRY

print('Imports OK.')
print(f'Available reward functions: {list(REWARD_REGISTRY.keys())}')

---
# Part 1: Reward Engineering

## Step 1.1: Review the Reward Functions

In [ ]:
# Inspect each reward function's source code
import inspect
from env import sparse_reward, dense_reward, velocity_only_reward, heavy_energy_reward

for name, fn in [('sparse', sparse_reward), ('dense', dense_reward),
                  ('velocity_only', velocity_only_reward), ('heavy_energy', heavy_energy_reward)]:
    print(f'--- {name} ---')
    print(inspect.getsource(fn))
    print()

### TODO 1.1 — Fill in this prediction table BEFORE running any experiments

| Reward function | Main incentive | Term most likely to cause problems | Your predicted behaviour |
|-----------------|----------------|------------------------------------|--------------------------|
| sparse | ??? | ??? | ??? |
| dense | ??? | ??? | ??? |
| velocity_only | ??? | ??? | ??? |
| heavy_energy | ??? | ??? | ??? |

## Step 1.2: Train with Each Reward Function

### Option A: Command line (recommended)

```bash
python train.py --reward sparse        --total_steps 300000 --run_name week2_sparse
python train.py --reward dense         --total_steps 300000 --run_name week2_dense
python train.py --reward velocity_only --total_steps 300000 --run_name week2_velocity_only
python train.py --reward heavy_energy  --total_steps 300000 --run_name week2_heavy_energy
```

Monitor:
```bash
tensorboard --logdir runs/
```

### Option B: Run in this notebook (cells below)

In [ ]:
def train_reward_run(reward_name, total_steps=300_000, n_envs=4, seed=0):
    """
    Train PPO on Walker2D with the given reward function.
    Returns the path to the saved evaluations.npz file.
    """
    run_dir = os.path.join('..', 'runs', f'week2_{reward_name}')
    os.makedirs(run_dir, exist_ok=True)

    reward_fn = get_reward_fn(reward_name)

    def make_env_fn(s):
        def _init():
            e = Walker2DEnv(render_mode=None, reward_fn=reward_fn)
            e.reset(seed=s)
            return e
        return _init

    train_env = make_vec_env(make_env_fn(seed), n_envs=n_envs)
    train_env = VecNormalize(train_env, norm_obs=True, norm_reward=True, clip_obs=10.0)

    model = PPO(
        'MlpPolicy', train_env,
        learning_rate=3e-4, n_steps=2048, batch_size=64,
        n_epochs=10, gamma=0.99, gae_lambda=0.95,
        clip_range=0.2, ent_coef=0.0,
        policy_kwargs=dict(net_arch=[256, 256]),
        verbose=1, tensorboard_log=run_dir,
    )

    eval_env = make_vec_env(make_env_fn(99), n_envs=1)
    eval_env = VecNormalize(eval_env, training=False, norm_obs=True, norm_reward=False, clip_obs=10.0)
    eval_env.obs_rms = train_env.obs_rms

    cb = EvalCallback(eval_env, best_model_save_path=run_dir, log_path=run_dir,
                      eval_freq=30_000, n_eval_episodes=10,
                      deterministic=True, verbose=0)

    model.learn(total_timesteps=total_steps, callback=cb,
                tb_log_name=reward_name, reset_num_timesteps=True)

    model.save(os.path.join(run_dir, 'final_model'))
    train_env.save(os.path.join(run_dir, 'vecnorm.pkl'))
    train_env.close(); eval_env.close()

    print(f'=== {reward_name} done ===')
    return os.path.join(run_dir, 'evaluations.npz')


print('Helper function defined.')

In [ ]:
# Train each reward function (sequentially)
REWARD_NAMES = ['sparse', 'dense', 'velocity_only', 'heavy_energy']
eval_paths = {}

for rname in REWARD_NAMES:
    print(f'\n{"="*50}')
    print(f'Training: {rname}')
    print('='*50)
    eval_paths[rname] = train_reward_run(rname, total_steps=300_000)

print('\nAll reward runs complete.')

## Step 1.3: Compare Training Curves

In [ ]:
# Load results (works whether you used Option A or B)
run_base = os.path.join('..', 'runs')
eval_paths = {
    name: os.path.join(run_base, f'week2_{name}', 'evaluations.npz')
    for name in REWARD_NAMES
}

colors = {'sparse': 'gray', 'dense': 'steelblue',
          'velocity_only': 'seagreen', 'heavy_energy': 'tomato'}

fig, ax = plt.subplots(figsize=(11, 5))

for name in REWARD_NAMES:
    path = eval_paths[name]
    if not os.path.exists(path):
        print(f'Not found (skipping): {path}')
        continue
    data  = np.load(path)
    ts    = data['timesteps']
    rews  = data['results'].mean(axis=1)
    stds  = data['results'].std(axis=1)
    ax.fill_between(ts, rews-stds, rews+stds, alpha=0.2, color=colors[name])
    ax.plot(ts, rews, label=name, color=colors[name], linewidth=2)

ax.set_xlabel('Training timesteps')
ax.set_ylabel('Mean episode reward (eval env)')
ax.set_title('Reward Function Comparison on Walker2D')
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(run_base, 'reward_comparison.png'), dpi=120)
plt.show()
print('Plot saved.')

In [ ]:
# Summary table
print(f'{"Reward Function":<20} {"Final Mean Reward":>20} {"Final Std":>12}')
print('-' * 55)

for name in REWARD_NAMES:
    path = eval_paths[name]
    if not os.path.exists(path):
        print(f'{name:<20} {"(not found)":>20}')
        continue
    data = np.load(path)
    last = data['results'][-1]
    print(f'{name:<20} {last.mean():>20.2f} {last.std():>12.2f}')

## Step 1.4: Describe What You Observed

### TODO 1.2 — For each reward function, write 2-4 sentences describing:
1. What the training curve looked like.
2. What behaviour the agent developed.
3. Any reward hacking or unexpected behaviour.

**sparse:**  
*Your answer here.*

**dense:**  
*Your answer here.*

**velocity_only:**  
*Your answer here.*

**heavy_energy:**  
*Your answer here.*

### TODO 1.3 — Record from your results:
- Which reward function produced the **highest** final reward?
- Which produced the **most stable** training curve (lowest std)?
- Did `sparse` learn at all?

---
# Part 2: Hyperparameter Ablation

## Step 2.1: Plan Your Experiments

### TODO 2.1 — Fill in this table BEFORE running. Write your prediction for each.

| Config name | What changed | Value | Your prediction |
|-------------|-------------|-------|------------------|
| baseline    | (nothing)   | defaults | steady learning |
| lr_small    | learning_rate | 1e-4 | ??? |
| lr_large    | learning_rate | 1e-3 | ??? |
| small_net   | net_arch | [64, 64] | ??? |
| short_rollout | n_steps | 512 | ??? |
| low_gamma   | gamma | 0.95 | ??? |
| ???         | ??? | ??? | ??? |

In [ ]:
# Define ablation configs
ABLATIONS = [
    ('week3_baseline',       {}),
    ('week3_lr_small',       {'learning_rate': 1e-4}),
    ('week3_lr_large',       {'learning_rate': 1e-3}),
    ('week3_small_net',      {'policy_kwargs': dict(net_arch=[64, 64])}),
    ('week3_short_rollout',  {'n_steps': 512, 'batch_size': 32}),
    ('week3_low_gamma',      {'gamma': 0.95}),
    # TODO 2.2: Add at least one more configuration of your own choosing.
    # Ideas:  ('week3_ent_mild', {'ent_coef': 0.01})
    #         ('week3_big_net',  {'policy_kwargs': dict(net_arch=[512, 512])})
    #         ('week3_long_rollout', {'n_steps': 4096, 'batch_size': 128})
]

print(f'Will run {len(ABLATIONS)} configurations:')
for name, cfg in ABLATIONS:
    print(f'  {name}: {cfg if cfg else "(baseline)"}')

## Step 2.2: Run Ablation Experiments

### Option A: Command line (recommended)

```bash
python train.py --run_name week3_baseline      --total_steps 300000
python train.py --run_name week3_lr_small      --total_steps 300000 --lr 0.0001
python train.py --run_name week3_lr_large      --total_steps 300000 --lr 0.001
python train.py --run_name week3_small_net     --total_steps 300000 --net_arch 64,64
python train.py --run_name week3_short_rollout --total_steps 300000 --n_steps 512
python train.py --run_name week3_low_gamma     --total_steps 300000 --gamma 0.95
```

### Option B: Run in this notebook

In [ ]:
def run_ablation(run_name, override_kwargs, total_steps=300_000, n_envs=4, seed=0):
    """
    Train Walker2D PPO with baseline config + overrides.
    Always uses dense reward (we're ablating hyperparameters, not rewards).
    """
    run_dir = os.path.join('..', 'runs', run_name)
    os.makedirs(run_dir, exist_ok=True)

    def make_env_fn(s):
        def _init():
            e = Walker2DEnv(render_mode=None)
            e.reset(seed=s)
            return e
        return _init

    train_env = make_vec_env(make_env_fn(seed), n_envs=n_envs)
    train_env = VecNormalize(train_env, norm_obs=True, norm_reward=True, clip_obs=10.0)

    baseline_kwargs = dict(
        policy='MlpPolicy', env=train_env,
        learning_rate=3e-4, n_steps=2048, batch_size=64,
        n_epochs=10, gamma=0.99, gae_lambda=0.95,
        clip_range=0.2, ent_coef=0.0,
        policy_kwargs=dict(net_arch=[256, 256]),
        verbose=1, tensorboard_log=run_dir,
    )
    baseline_kwargs.update(override_kwargs)

    model = PPO(**baseline_kwargs)

    eval_env = make_vec_env(make_env_fn(99), n_envs=1)
    eval_env = VecNormalize(eval_env, training=False,
                            norm_obs=True, norm_reward=False, clip_obs=10.0)
    eval_env.obs_rms = train_env.obs_rms

    cb = EvalCallback(eval_env, best_model_save_path=run_dir, log_path=run_dir,
                      eval_freq=30_000, n_eval_episodes=10,
                      deterministic=True, verbose=0)

    model.learn(total_timesteps=total_steps, callback=cb,
                tb_log_name=run_name, reset_num_timesteps=True)

    model.save(os.path.join(run_dir, 'final_model'))
    train_env.save(os.path.join(run_dir, 'vecnorm.pkl'))
    train_env.close(); eval_env.close()

    data = np.load(os.path.join(run_dir, 'evaluations.npz'))
    return data['timesteps'], data['results'].mean(axis=1), data['results'].std(axis=1)


print('Ablation function defined.')

In [ ]:
# Run all ablations sequentially
ablation_results = {}
for run_name, overrides in ABLATIONS:
    print(f'\n{"="*60}')
    print(f'Running: {run_name}')
    print('='*60)
    ts, means, stds = run_ablation(run_name, overrides)
    ablation_results[run_name] = (ts, means, stds)
    print(f'Final mean reward: {means[-1]:.2f}')

print('\nAll ablations complete.')

## Step 2.3: Compare Ablation Curves

In [ ]:
# Load results from disk
run_names = [name for name, _ in ABLATIONS]

loaded = {}
for name in run_names:
    path = os.path.join(run_base, name, 'evaluations.npz')
    if os.path.exists(path):
        data = np.load(path)
        loaded[name] = (data['timesteps'],
                        data['results'].mean(axis=1),
                        data['results'].std(axis=1))
    else:
        print(f'Not found (skipping): {path}')

print(f'Loaded {len(loaded)} runs.')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
cmap_colors = plt.cm.tab10(np.linspace(0, 1, len(loaded)))

for (name, (ts, means, stds)), color in zip(loaded.items(), cmap_colors):
    label = name.replace('week3_', '')
    ax.fill_between(ts, means-stds, means+stds, alpha=0.15, color=color)
    ax.plot(ts, means, label=label, color=color, linewidth=2)

ax.set_xlabel('Training timesteps')
ax.set_ylabel('Mean episode reward')
ax.set_title('Hyperparameter Ablation on Walker2D')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.4)
plt.tight_layout()
save_path = os.path.join(run_base, 'ablation_comparison.png')
plt.savefig(save_path, dpi=120)
plt.show()
print('Saved:', save_path)

In [ ]:
# Summary table
print(f'{"Config":<25} {"Final Reward":>14} {"Std":>8}')
print('-' * 50)

rows = []
for name, (ts, means, stds) in loaded.items():
    rows.append((name, means[-1], stds[-1]))

for name, m, s in sorted(rows, key=lambda x: -x[1]):
    label = name.replace('week3_', '')
    print(f'{label:<25} {m:>14.2f} {s:>8.2f}')

### TODO 2.3 — Paste the table into your report.

### TODO 2.4 — Which single parameter had the biggest impact on performance? Why?

*Write 3-5 sentences here.*

### TODO 2.5 — Choose the Best Configuration for Week 4

**Best configuration:**

- `learning_rate`: ???
- `net_arch`: ???
- `n_steps`: ???
- `gamma`: ???
- `reward function` (from Part 1): ???

**Justification** (2-4 sentences):  
*Write here.*

---
# Part 3: Custom Reward Function

### TODO 3.1 — Before writing code, answer:

1. **What behaviour are you trying to encourage?**  
   *Your answer.*

2. **What term(s) are you adding/modifying?**  
   *Your answer.*

3. **Mathematical formula:** $r_t = \ldots$

4. **Expected magnitude of each term?**  
   *Your answer.*

In [ ]:
# TODO 3.2 — Implement your custom reward function.
#
# Observation cheat sheet:
#   obs[0]  = torso height (m)     obs[14] = forward velocity v_x
#   obs[1]  = torso pitch (rad)    obs[15] = lateral velocity v_y
#   obs[2:8]  = joint angles       obs[16] = vertical velocity v_z
#   obs[8:14] = joint velocities   obs[17] = pitch rate
#   obs[18] = left foot contact    obs[19] = right foot contact
#
# Ideas:  smoothness penalty, symmetry bonus, velocity target, pitch stability

def custom_reward(obs, action, info):
    """Your custom reward. Start from dense and add/modify at least one component."""
    forward  = 1.0 * np.clip(obs[14], 0.0, 5.0)
    alive    = 1.0
    energy   = 0.001 * np.sum(np.square(action))
    lateral  = 0.5 * abs(obs[15])
    height   = 0.1 * max(0.0, 1.0 - abs(obs[0] - 1.2))

    # TODO: Add your custom term(s) below
    # custom_term = ...

    return float(
        forward + alive - energy - lateral + height
        # + custom_term
    )

In [ ]:
# TODO 3.3 — Sanity-check your reward by hand.

obs_walking = np.zeros(22)
obs_walking[0] = 1.2; obs_walking[14] = 2.0
obs_walking[18] = 1.0; obs_walking[19] = 0.0
action_mod = np.array([0.3, -0.2, 0.1, -0.3, 0.2, -0.1])

obs_still = np.zeros(22); obs_still[0] = 1.2
action_zero = np.zeros(6)

info_test = {'fell': False, 'dx': 0.03, 'prev_action': np.zeros(6)}

r_walk = custom_reward(obs_walking, action_mod, info_test)
r_still = custom_reward(obs_still, action_zero, info_test)

print(f'Walking reward:  {r_walk:.4f}')
print(f'Standing still:  {r_still:.4f}')
print('Good!' if r_walk > r_still else 'WARNING: standing >= walking. Check your terms!')

---
# Reflection Questions

## Reward Engineering

**Q1.** Which reward function would you choose for actual walking? Justify using your numbers.

**Q2.** The `velocity_only` reward removes the energy penalty. What did the agent trade off?

**Q3.** The `heavy_energy` penalty is only 10x the default. Did this small change have a large effect? Why?

**Q4.** Suppose you wanted to encourage regular, metronome-like steps. What observation info would you use? Write a rough formula.

**Q5.** What is one limitation of comparing reward functions only by training curves? What else would you measure?

## Hyperparameter Ablation

**Q6.** Which hyperparameter had the largest effect? Was this what you predicted?

**Q7.** If you ran each config with 3 different seeds, how would that change your conclusions? Why is this important?

**Q8.** `approx_kl` measures how much the policy changed each update. For which config was it largest? What does that suggest?

**Q9.** If mentoring a new student, which hyperparameter would you tell them to tune first? Explain.

**Q10.** You changed LR *and* network size at the same time and got better results. Can you conclude both helped? Why or why not?

---
# Checklist

### End of Week 2:
- [ ] Part 1 cells run, TODO 1.1-1.3 answered.
- [ ] Comparison plot `runs/reward_comparison.png`.
- [ ] Summary table with final mean rewards.
- [ ] Be ready to explain reward hacking.

### End of Week 3:
- [ ] Part 2 cells run, TODO 2.1-2.5 answered.
- [ ] Ablation plot `runs/ablation_comparison.png`.
- [ ] Part 3: custom reward with sanity check.
- [ ] All reflection questions answered (Q1-Q10).
- [ ] Best config for Week 4 clearly stated.